# Lab Work - 11.1

---

## Dataset (from Q1)

| Index | Age | Hours Studied | Score |
|-------|-----|---------------|-------|
| 0     | 25  | 1             | 50    |
| 1     | 30  | 3             | 65    |
| 2     | 35  | 2             | 60    |
| 3     | 40  | 5             | 85    |
| 4     | 45  | 4             | 80    |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Fixed seed for reproducibility
np.random.seed(42)

# Original dataset
data = pd.DataFrame({
    'Age': [25, 30, 35, 40, 45],
    'Hours_Studied': [1, 3, 2, 5, 4],
    'Score': [50, 65, 60, 85, 80]
})
data.index.name = 'Index'
print("Original Dataset:")
display(data)

---
# Q1 – Bootstrap Sampling by Hand

### 01 & 02 – Create two Bootstrap Samples

We draw **5 indices with replacement** from `{0, 1, 2, 3, 4}`.

In [ ]:
# Bootstrap Sample 1
boot1_indices = np.random.choice([0, 1, 2, 3, 4], size=5, replace=True)
boot1 = data.iloc[boot1_indices].copy()
boot1['Original_Index'] = boot1_indices

print("Bootstrap Sample 1 – Selected Indices:", boot1_indices.tolist())
print("\nBootstrap Sample 1:")
display(boot1)

# Bootstrap Sample 2 (fresh draw)
boot2_indices = np.random.choice([0, 1, 2, 3, 4], size=5, replace=True)
boot2 = data.iloc[boot2_indices].copy()
boot2['Original_Index'] = boot2_indices

print("\nBootstrap Sample 2 – Selected Indices:", boot2_indices.tolist())
print("\nBootstrap Sample 2:")
display(boot2)

### 04 – Compute mean Score for each bootstrap sample (simple prediction of each “tree”)

In [ ]:
mean1 = boot1['Score'].mean()
mean2 = boot2['Score'].mean()

print(f"Tree 1 (Bootstrap Sample 1) mean Score prediction: {mean1:.2f}")
print(f"Tree 2 (Bootstrap Sample 2) mean Score prediction: {mean2:.2f}")

### 05 – Aggregate – average both tree predictions (ensemble output)

In [ ]:
ensemble_pred = (mean1 + mean2) / 2
print(f"Final Ensemble Prediction (average of both trees): {ensemble_pred:.2f}")

### 06 – Identify Out-of-Bag (OOB) samples

OOB samples are the original indices that were **never selected** in a particular bootstrap sample.

In [ ]:
all_indices = set(range(5))

oob1 = sorted(list(all_indices - set(boot1_indices)))
oob2 = sorted(list(all_indices - set(boot2_indices)))

print(f"OOB indices for Bootstrap Sample 1: {oob1}")
print(f"OOB indices for Bootstrap Sample 2: {oob2}")

print("\nOOB rows for Sample 1:")
display(data.iloc[oob1] if oob1 else "None (all points were selected)")

print("\nOOB rows for Sample 2:")
display(data.iloc[oob2] if oob2 else "None (all points were selected)")

---
# Q2 – Decision Stumps & Random Feature Selection

### 01 – Use the full dataset (no bootstrap for the stumps themselves)

We deliberately **do not** use all features at each split (random subspace method).

### 02 – Stump A (feature = Age)

Split: **Age ≤ 35** vs **Age > 35**  
Compute mean Score for each group.

In [ ]:
# Stump A – Age
left_A = data[data['Age'] <= 35]
right_A = data[data['Age'] > 35]

mean_left_A = left_A['Score'].mean()
mean_right_A = right_A['Score'].mean()

print("=== Stump A (Feature: Age) ===")
print(f"Split: Age ≤ 35  →  mean Score = {mean_left_A:.2f}")
print(f"Split: Age > 35  →  mean Score = {mean_right_A:.2f}")
print("\nLeft group (Age ≤ 35):")
display(left_A)
print("Right group (Age > 35):")
display(right_A)

### 03 – Stump B (feature = Hours Studied)

Split: **Hours ≤ 3** vs **Hours > 3**  
Compute mean Score for each group.

In [ ]:
# Stump B – Hours Studied
left_B = data[data['Hours_Studied'] <= 3]
right_B = data[data['Hours_Studied'] > 3]

mean_left_B = left_B['Score'].mean()
mean_right_B = right_B['Score'].mean()

print("=== Stump B (Feature: Hours Studied) ===")
print(f"Split: Hours ≤ 3  →  mean Score = {mean_left_B:.2f}")
print(f"Split: Hours > 3  →  mean Score = {mean_right_B:.2f}")
print("\nLeft group (Hours ≤ 3):")
display(left_B)
print("Right group (Hours > 3):")
display(right_B)

### 04 – Regression prediction for new point [Age = 38, Hours = 4]

Get each stump’s leaf value, then average for the Random Forest output.

In [ ]:
new_point = {'Age': 38, 'Hours_Studied': 4}

# Stump A prediction
pred_A = mean_left_A if new_point['Age'] <= 35 else mean_right_A

# Stump B prediction
pred_B = mean_left_B if new_point['Hours_Studied'] <= 3 else mean_right_B

rf_regression = (pred_A + pred_B) / 2

print(f"New point: Age = {new_point['Age']}, Hours Studied = {new_point['Hours_Studied']}")
print(f"\nStump A (Age) prediction: {pred_A:.2f}")
print(f"Stump B (Hours) prediction: {pred_B:.2f}")
print(f"\nRandom Forest Regression output (average): {rf_regression:.2f}")

### 05 – Classification scenario

Label: Score ≥ 70 → **Pass**, Score < 70 → **Fail**  
Apply majority vote across both stumps for the same new point.

In [ ]:
def to_class(score):
    return 'Pass' if score >= 70 else 'Fail'

class_A = to_class(pred_A)
class_B = to_class(pred_B)

# Majority vote
votes = [class_A, class_B]
majority = max(set(votes), key=votes.count)

print(f"Stump A class: {class_A} (from score {pred_A:.2f})")
print(f"Stump B class: {class_B} (from score {pred_B:.2f})")
print(f"\nMajority Vote (Random Forest Classification): {majority}")

### 06 – Reflection

**Why does using a random subset of features at each split reduce correlation between trees?**

> When every tree is allowed to consider **all** features, the strongest features tend to be selected near the root of almost every tree. Consequently the trees become highly correlated (they make similar mistakes). By forcing each split to choose from a random subset of features, different trees are more likely to use different variables, increasing diversity. Averaging diverse trees reduces variance more effectively than averaging highly correlated ones.

---
# Q3 – Visualize the Forest

### 01 & 02 – Draw both decision stumps side-by-side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Stump A (Age) ----
ax = axes[0]
ax.set_title('Stump A – Feature: Age\nSplit at Age ≤ 35 vs Age > 35', fontsize=12, fontweight='bold')

# Draw a simple tree diagram
ax.plot([0.5, 0.5], [0.8, 0.55], 'k-', lw=2)
ax.plot([0.5, 0.25], [0.55, 0.3], 'k-', lw=2)
ax.plot([0.5, 0.75], [0.55, 0.3], 'k-', lw=2)

# Nodes
ax.add_patch(plt.Circle((0.5, 0.85), 0.08, color='skyblue', ec='black', zorder=5))
ax.text(0.5, 0.85, 'Age', ha='center', va='center', fontweight='bold', fontsize=10)

ax.add_patch(plt.Circle((0.25, 0.25), 0.1, color='lightgreen', ec='black', zorder=5))
ax.text(0.25, 0.25, f'≤35\nμ={mean_left_A:.1f}', ha='center', va='center', fontsize=9)

ax.add_patch(plt.Circle((0.75, 0.25), 0.1, color='salmon', ec='black', zorder=5))
ax.text(0.75, 0.25, f'>35\nμ={mean_right_A:.1f}', ha='center', va='center', fontsize=9)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# ---- Stump B (Hours) ----
ax = axes[1]
ax.set_title('Stump B – Feature: Hours Studied\nSplit at Hours ≤ 3 vs Hours > 3', fontsize=12, fontweight='bold')

ax.plot([0.5, 0.5], [0.8, 0.55], 'k-', lw=2)
ax.plot([0.5, 0.25], [0.55, 0.3], 'k-', lw=2)
ax.plot([0.5, 0.75], [0.55, 0.3], 'k-', lw=2)

ax.add_patch(plt.Circle((0.5, 0.85), 0.08, color='skyblue', ec='black', zorder=5))
ax.text(0.5, 0.85, 'Hours', ha='center', va='center', fontweight='bold', fontsize=10)

ax.add_patch(plt.Circle((0.25, 0.25), 0.1, color='lightgreen', ec='black', zorder=5))
ax.text(0.25, 0.25, f'≤3\nμ={mean_left_B:.1f}', ha='center', va='center', fontsize=9)

ax.add_patch(plt.Circle((0.75, 0.25), 0.1, color='salmon', ec='black', zorder=5))
ax.text(0.75, 0.25, f'>3\nμ={mean_right_B:.1f}', ha='center', va='center', fontsize=9)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

### 03 – Mark OOB samples

We show which original data points were excluded from each bootstrap sample.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, oob, title in zip(axes, [oob1, oob2],
                          ['Bootstrap Sample 1 – OOB', 'Bootstrap Sample 2 – OOB']):
    colors = ['red' if i in oob else 'steelblue' for i in range(5)]
    markers = ['X' if i in oob else 'o' for i in range(5)]
    
    for i in range(5):
        ax.scatter(data.loc[i, 'Age'], data.loc[i, 'Hours_Studied'],
                   c=colors[i], s=180, marker=markers[i],
                   edgecolors='black', linewidths=1.2, zorder=5)
        ax.text(data.loc[i, 'Age']+0.4, data.loc[i, 'Hours_Studied']+0.15,
                f'idx {i}\nScore={data.loc[i,"Score"]}', fontsize=8)
    
    ax.set_xlabel('Age')
    ax.set_ylabel('Hours Studied')
    ax.set_title(title)
    ax.set_xlim(22, 48)
    ax.set_ylim(0, 6)
    
    legend_elements = [
        Patch(facecolor='steelblue', edgecolor='black', label='In-bag'),
        Patch(facecolor='red', edgecolor='black', label='OOB (excluded)')
    ]
    ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

print(f"OOB Sample 1 indices: {oob1}")
print(f"OOB Sample 2 indices: {oob2}")

### 04 – Feature Importance bar chart

Importance is proportional to the number of splits each feature contributed (here each feature was used once).

In [ ]:
features = ['Age', 'Hours Studied']
importance = [1, 1]   # each feature used in exactly one stump

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(features, importance, color=['#4C72B0', '#55A868'], edgecolor='black')
ax.set_xlabel('Number of splits (Importance)')
ax.set_title('Feature Importance (based on number of splits)')
ax.set_xlim(0, 1.5)

for bar, val in zip(bars, importance):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 05 – Bonus: 2D scatter plot with prediction regions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Colour points by class (Pass / Fail)
classes = ['Pass' if s >= 70 else 'Fail' for s in data['Score']]
colors_map = {'Pass': 'green', 'Fail': 'red'}

for i, row in data.iterrows():
    ax.scatter(row['Age'], row['Hours_Studied'],
               c=colors_map[classes[i]], s=200, edgecolors='black',
               linewidths=1.5, zorder=5)
    ax.text(row['Age']+0.5, row['Hours_Studied']+0.15,
            f'{row["Score"]}', fontsize=9)

# Draw split boundaries
ax.axvline(x=35, color='blue', linestyle='--', lw=2, label='Age split (≤35)')
ax.axhline(y=3, color='purple', linestyle='--', lw=2, label='Hours split (≤3)')

# Shade regions lightly (approximate)
ax.axvspan(20, 35, alpha=0.08, color='blue')
ax.axvspan(35, 50, alpha=0.08, color='orange')
ax.axhspan(0, 3, alpha=0.05, color='purple')
ax.axhspan(3, 7, alpha=0.05, color='green')

ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Hours Studied', fontsize=12)
ax.set_title('Bonus: Scatter plot (Age vs Hours)\ncoloured by class + stump split boundaries', fontsize=13)
ax.set_xlim(22, 48)
ax.set_ylim(0, 6.5)

# Custom legend
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green',
               markersize=12, label='Pass (≥70)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
               markersize=12, label='Fail (<70)'),
    plt.Line2D([0], [0], color='blue', linestyle='--', lw=2, label='Age ≤ 35 split'),
    plt.Line2D([0], [0], color='purple', linestyle='--', lw=2, label='Hours ≤ 3 split')
]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

---
# Q4 – Explain It Cold

### 01 – What is Bagging (Bootstrap Aggregating)?

**In my own words:**

Bagging works in three simple steps:

1. **Sample** – Create many new training sets by drawing observations *with replacement* from the original data (bootstrap samples). Each sample is roughly the same size as the original dataset but contains duplicates and leaves some points out.
2. **Train** – Fit a model (usually a decision tree) independently on each of these bootstrap samples.
3. **Aggregate** – Combine the predictions of all the models. For regression we take the average; for classification we take a majority vote.

The goal is to reduce the variance of unstable learners (like deep trees) without increasing bias much.

### 02 – Why does averaging multiple weak learners outperform a single strong learner?

This is the classic **bias-variance tradeoff** argument.

- A single complex model (deep tree, high-capacity neural net) can have **low bias** but **high variance** – it overfits the particular training sample and changes a lot if the data changes slightly.
- Many weak / shallow models each have higher bias but much lower variance.
- When we average many such models that are trained on different bootstrap samples, the random errors tend to cancel out. The variance of the ensemble drops roughly by a factor of the number of models (if they are uncorrelated), while the bias stays roughly the same as that of the individual weak learners.

Result: lower total error than a single overfitted strong learner.

### 03 – Difference between Random Forest for regression vs classification

| Aspect              | Regression                              | Classification                          |
|---------------------|-----------------------------------------|-----------------------------------------|
| Aggregation method  | **Average** the numeric predictions of all trees | **Majority vote** (mode) of the class labels |
| When it applies     | Target is continuous (e.g. Score, price, temperature) | Target is discrete / categorical (e.g. Pass/Fail, disease yes/no) |
| Output of each tree | A real number (leaf mean)               | A class label                           |
| Final output        | Continuous value                        | Class label (sometimes with vote percentages) |

Both use the same underlying idea (bootstrap + random features), only the way predictions are combined changes.

### 04 – What is OOB error and why Random Forest does not strictly need a separate validation set

**Out-of-Bag (OOB) error**

- For every bootstrap sample, some original observations are left out (on average about 37% of the data).
- These left-out points are called the **out-of-bag** samples for that tree.
- After all trees are grown, each original observation can be predicted using **only the trees that did not see it** during training.
- The average error of these OOB predictions is the **OOB error estimate**.

**Why no separate validation set is required**

Because every data point is automatically held out of a substantial fraction of the trees, the OOB predictions act as an almost unbiased estimate of the true generalisation error. This is computationally free (no extra models need to be trained) and uses the data more efficiently than a traditional train/validation split, especially when the dataset is small.
